# Сбор данных о кофейнях Алматы из 2ГИС

Ноутбук воспроизводит сетку из исходного `03_2gis_data.ipynb` по локальным данным и содержит сборщик, mock-тесты и экспорт в GeoParquet. При запуске всех ячеек с настройками по умолчанию выполняются только локальные операции и тесты.

- Для тестов нужен `requests`. Сборщик использует SQLite и блокировку файла на macOS/Linux.
- Для подготовки сетки и экспорта нужны `geopandas`, `shapely`, `numpy` и `pyarrow`. Используйте окружение проекта.
- Для сетки нужны локальный кэш границы Алматы в `notebooks/cache` и `data/processed/almaty_features.parquet`. Укажите путь к каталогу проекта в разделе настроек, если открываете ноутбук отдельно.

[Пагинация и ограничения 2ГИС](https://docs.2gis.com/api/search/places/examples/results) · [Коды ответа 2ГИС](https://docs.2gis.com/api/search/places/troubleshooting)


## Сборщик

Прогресс сохраняется после каждой проверенной страницы. SQLite хранит уникальные организации, следующую страницу каждого центра и суммарное число HTTP-попыток, включая повторы. При повторном запуске используйте тот же файл прогресса и те же параметры поиска. Для изменённой сетки, радиуса, запроса или пагинации укажите новый файл.


In [1]:
from contextlib import closing, contextmanager
from dataclasses import dataclass
from datetime import timezone
from email.utils import parsedate_to_datetime
import fcntl
import json
import math
from pathlib import Path
import random
import sqlite3
import time

import requests

PLACES_URL = "https://catalog.api.2gis.com/3.0/items"


class CollectionStopped(RuntimeError):
    """Остановка, после которой сохранённый прогресс можно использовать снова."""


@dataclass(frozen=True)
class Search:
    query: str = "кофейня"
    radius_m: int = 1000
    page_size: int = 10
    max_pages: int = 5

    def __post_init__(self):
        if (not isinstance(self.query, str) or not self.query.strip()
                or type(self.radius_m) is not int or not 1 <= self.radius_m <= 50_000):
            raise ValueError("Проверьте текст поиска и радиус.")
        if (type(self.page_size) is not int or not 1 <= self.page_size <= 50
                or type(self.max_pages) is not int or not 1 <= self.max_pages <= 1_000_000):
            raise ValueError("Проверьте размер страницы и предел страниц.")

    def params(self):
        return {
            "q": self.query, "type": "branch", "radius": self.radius_m,
            "page_size": self.page_size, "fields": "items.point,items.reviews",
        }


def retry_after_seconds(value, now):
    """Retry-After может содержать число секунд или дату HTTP."""
    if not value:
        return 0
    value = value.strip()
    try:
        if value.isascii() and value.isdecimal():
            return int(value) if len(value) <= 12 else math.inf
        retry_at = parsedate_to_datetime(value)
        if retry_at.tzinfo is None:
            retry_at = retry_at.replace(tzinfo=timezone.utc)
        return max(0, retry_at.timestamp() - now)
    except (TypeError, ValueError, OverflowError):
        return 0


class RequestBudget:
    """Лимит всех попыток и самое раннее время следующего запроса."""

    def __init__(self, db, limit, interval, *, now=time.time, sleep=time.sleep):
        self.db = db
        self.limit = limit
        self.interval = interval
        self.now = now
        self.sleep = sleep

    def defer(self, deadline):
        with self.db:
            self.db.execute(
                "UPDATE run SET next_at = max(next_at, ?)", (deadline,)
            )

    def reserve(self):
        attempts, next_at = self.db.execute(
            "SELECT attempts, next_at FROM run"
        ).fetchone()
        if attempts >= self.limit:
            raise CollectionStopped("Достигнут суммарный лимит HTTP-попыток.")
        delay = next_at - self.now()
        if delay > 60:
            raise CollectionStopped("Пауза после временной ошибки ещё действует.")
        while delay > 0:
            self.sleep(delay)
            delay = next_at - self.now()
        # Списываем попытку до отправки. При прерывании счётчик консервативен.
        with self.db:
            self.db.execute(
                "UPDATE run SET attempts = attempts + 1, next_at = ?",
                (self.now() + self.interval,),
            )


def get_with_retry(get, params, budget, max_attempts=3):
    """Проверяет HTTP и meta.code, возвращает JSON успешного ответа."""
    for attempt in range(max_attempts):
        budget.reserve()
        retry_after = None
        try:
            response = get(PLACES_URL, params=params, timeout=(5, 15),
                           allow_redirects=False)
        except (requests.exceptions.SSLError, requests.exceptions.ProxyError):
            raise CollectionStopped("Ошибка TLS или прокси.") from None
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            reason = "временный сетевой сбой"
        except requests.exceptions.RequestException:
            raise CollectionStopped("Ошибка при отправке HTTP-запроса.") from None
        else:
            status = response.status_code
            retry_after = response.headers.get("Retry-After")
            if status == 200:
                try:
                    data = response.json()
                except ValueError:
                    raise CollectionStopped("Ответ 2ГИС не является JSON.") from None
                meta = data.get("meta") if isinstance(data, dict) else None
                code = meta.get("code") if isinstance(meta, dict) else None
                if type(code) is not int:
                    raise CollectionStopped("В ответе 2ГИС отсутствует числовой meta.code.")
                if code == 200:
                    return data
                error = meta.get("error")
                if (code == 404 and isinstance(error, dict)
                        and error.get("type") == "itemNotFound"):
                    return {"result": {"items": [], "total": 0}}
                if code not in (408, 429) and not 500 <= code <= 599:
                    raise CollectionStopped(f"2ГИС вернул meta.code {code}.")
                reason = f"meta.code {code}"
            else:
                if status not in (408, 429) and not 500 <= status <= 599:
                    raise CollectionStopped(f"2ГИС вернул HTTP {status}.")
                reason = f"HTTP {status}"
        now = budget.now()
        server_delay = retry_after_seconds(retry_after, now)
        delay = max(server_delay, 2 ** attempt + random.uniform(0, 0.5))
        budget.defer(now + delay)
        if attempt + 1 == max_attempts or delay > 60:
            raise CollectionStopped(f"Пауза в сборе 2ГИС ({reason}).")


def page_items(data, page_size):
    result = data.get("result")
    total = result.get("total") if isinstance(result, dict) else None
    if type(total) is not int or total < 0:
        raise CollectionStopped("В ответе 2ГИС отсутствует корректный result.total.")
    items = result.get("items", [] if total == 0 else None)
    if not isinstance(items, list) or len(items) > min(total, page_size):
        raise CollectionStopped("Некорректный список result.items.")
    for item in items:
        item_id = item.get("id") if isinstance(item, dict) else None
        if not isinstance(item_id, str) or not item_id.strip():
            raise CollectionStopped("В result.items есть объект без строкового ID.")
    return total, items


SCHEMA = """
CREATE TABLE IF NOT EXISTS run (
    id INTEGER PRIMARY KEY CHECK (id = 1),
    plan TEXT NOT NULL,
    attempts INTEGER NOT NULL DEFAULT 0,
    next_at REAL NOT NULL DEFAULT 0
);
CREATE TABLE IF NOT EXISTS centers (
    point TEXT PRIMARY KEY,
    next_page INTEGER NOT NULL DEFAULT 1,
    expected_total INTEGER,
    status TEXT NOT NULL DEFAULT 'pending'
);
CREATE TABLE IF NOT EXISTS center_items (
    point TEXT NOT NULL,
    organization_id TEXT NOT NULL,
    PRIMARY KEY (point, organization_id)
);
CREATE TABLE IF NOT EXISTS organizations (
    id TEXT PRIMARY KEY,
    name TEXT,
    lon REAL,
    lat REAL,
    general_rating REAL,
    general_review_count INTEGER
);
"""


@contextmanager
def open_checkpoint(path):
    """Один сборщик на файл. Блокировка освобождается при завершении процесса."""
    path = Path(path).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.with_suffix(path.suffix + ".lock").open("a") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise CollectionStopped("Этот файл прогресса уже используется.") from None
        with closing(sqlite3.connect(path)) as db:
            db.executescript(SCHEMA)
            yield db


def prepare_checkpoint(db, points, search):
    plan = json.dumps(
        {"version": 1, "url": PLACES_URL, "params": search.params(),
         "max_pages": search.max_pages, "points": points},
        ensure_ascii=False, sort_keys=True,
    )
    existing = db.execute("SELECT plan FROM run").fetchone()
    if existing and existing[0] != plan:
        raise CollectionStopped("План поиска изменился. Выберите новый файл прогресса.")
    if not existing:
        with db:
            db.execute("INSERT INTO run (id, plan) VALUES (1, ?)", (plan,))
            db.executemany("INSERT INTO centers (point) VALUES (?)", [(p,) for p in points])


def organization_row(item):
    point = item.get("point")
    lon, lat = (point.get("lon"), point.get("lat")) if isinstance(point, dict) else (None, None)
    if not (
        type(lon) in (int, float) and -180 <= lon <= 180
        and type(lat) in (int, float) and -90 <= lat <= 90
    ):
        lon, lat = None, None
    reviews = item.get("reviews")
    reviews = reviews if isinstance(reviews, dict) else {}
    rating = reviews.get("general_rating")
    count = reviews.get("general_review_count")
    return (
        item["id"], item.get("name") if isinstance(item.get("name"), str) else None,
        lon, lat,
        rating if type(rating) in (int, float) and 0 <= rating <= 1e6 else None,
        count if type(count) is int and 0 <= count < 2**63 else None,
    )


def fetch_coffee_shops(db, point, search, key, get, budget):
    page, expected_total, status = db.execute(
        "SELECT next_page, expected_total, status FROM centers WHERE point = ?",
        (point,),
    ).fetchone()
    while status == "pending":
        params = dict(search.params(), key=key, point=point, page=page)
        total, items = page_items(get_with_retry(get, params, budget), search.page_size)
        # Если выдача меняется во время сбора, учитываем наибольший заявленный total.
        expected_total = max(total, expected_total or 0)
        # Записи страницы и курсор фиксируются одной транзакцией.
        with db:
            db.executemany(
                """INSERT INTO organizations VALUES (?, ?, ?, ?, ?, ?)
                   ON CONFLICT(id) DO UPDATE SET
                       name = coalesce(excluded.name, organizations.name),
                       lon = coalesce(excluded.lon, organizations.lon),
                       lat = coalesce(excluded.lat, organizations.lat),
                       general_rating = coalesce(excluded.general_rating,
                                                 organizations.general_rating),
                       general_review_count = coalesce(excluded.general_review_count,
                                                       organizations.general_review_count)""",
                map(organization_row, items),
            )
            db.executemany(
                "INSERT OR IGNORE INTO center_items VALUES (?, ?)",
                ((point, item["id"]) for item in items),
            )
            seen = db.execute(
                "SELECT count(*) FROM center_items WHERE point = ?", (point,)
            ).fetchone()[0]
            if seen >= expected_total:
                status = "complete"
            elif not items or page * search.page_size >= expected_total:
                status = "short"
            elif page >= search.max_pages:
                status = "capped"
            db.execute(
                "UPDATE centers SET next_page = ?, expected_total = ?, status = ? WHERE point = ?",
                (page + 1, expected_total, status, point),
            )
        page += 1


def checkpoint_summary(db):
    statuses = {status: 0 for status in ("pending", "complete", "capped", "short")}
    statuses.update(db.execute("SELECT status, count(*) FROM centers GROUP BY status"))
    attempts = db.execute("SELECT attempts FROM run").fetchone()[0]
    total, missing = db.execute(
        "SELECT count(*), count(*) - count(lon) FROM organizations"
    ).fetchone()
    return {"attempts": attempts, "centers": statuses,
            "organizations": total, "without_coordinates": missing}


def collect(points, search, checkpoint, *, key, max_requests_total, center_limit, get,
            interval=1.0, now=time.time, sleep=time.sleep):
    points = tuple(dict.fromkeys(points))
    if not key or not points:
        raise ValueError("Для сбора нужны ключ и центры поиска.")
    if type(max_requests_total) is not int or max_requests_total < 1:
        raise ValueError("Укажите положительный суммарный лимит HTTP-попыток.")
    if type(center_limit) is not int or not 1 <= center_limit <= len(points):
        raise ValueError("Укажите число центров от 1 до размера сетки.")
    if not math.isfinite(interval) or not 1 <= interval <= 60:
        raise ValueError("Пауза между запросами должна составлять от 1 до 60 секунд.")
    with open_checkpoint(checkpoint) as db:
        prepare_checkpoint(db, points, search)
        budget = RequestBudget(db, max_requests_total, interval, now=now, sleep=sleep)
        stop_reason = None
        for index, point in enumerate(points[:center_limit], start=1):
            try:
                fetch_coffee_shops(db, point, search, key, get, budget)
            except CollectionStopped as error:
                stop_reason = f"Центр {index}/{len(points)}. {error}"
                break
        return checkpoint_summary(db) | {"stop_reason": stop_reason}

## Подготовка сетки и экспорт

In [2]:
def prepare_almaty_grid(project_root):
    from itertools import product

    import geopandas as gpd
    import numpy as np
    from shapely.geometry import Point, shape

    root = Path(project_root)
    cache = root / "notebooks" / "cache"
    city = None
    for path in sorted(cache.glob("*.json")):
        cached = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(cached, list):
            continue
        for place in cached:
            if not isinstance(place, dict):
                continue
            geojson = place.get("geojson") or {}
            if (place.get("display_name") == "Almaty, Kazakhstan"
                    and geojson.get("type") in {"Polygon", "MultiPolygon"}):
                city = shape(geojson)
                break
        if city is not None:
            break
    if city is None:
        raise FileNotFoundError("В notebooks/cache нужна сохранённая граница «Almaty, Kazakhstan».")

    almaty = gpd.GeoSeries([city], crs="EPSG:4326")
    almaty_m = almaty.to_crs(almaty.estimate_utm_crs())
    min_x, min_y, max_x, max_y = almaty_m.total_bounds
    grid = gpd.GeoDataFrame(
        geometry=[Point(x, y) for x, y in product(
            np.arange(min_x, max_x, 1000), np.arange(min_y, max_y, 1000)
        )],
        crs=almaty_m.crs,
    )
    inside = grid[grid.geometry.within(almaty_m.iloc[0])]
    features = gpd.read_parquet(root / "data" / "processed" / "almaty_features.parquet")
    features = features.to_crs(inside.crs)
    columns = [
        "houses_500m", "apartments_500m", "office_polygons_500m",
        "offices_pois_500m", "universities_500m", "cafes_500m",
        "bus_stops_500m", "schools_500m", "colleges_500m",
    ]
    matched = gpd.sjoin_nearest(inside, features[columns + ["geometry"]], how="left")
    selected = matched.loc[(matched[columns] > 0).any(axis=1)]
    # Ближайшие локации на равном расстоянии могут дать несколько строк на один центр.
    selected = selected.loc[~selected.index.duplicated()].sort_index()
    search_lonlat = selected.to_crs("EPSG:4326").geometry
    points = list(dict.fromkeys(
        f"{point.x:.8f},{point.y:.8f}" for point in search_lonlat
    ))
    area = selected.geometry.buffer(1000).union_all()
    stats = {
        "grid_inside_almaty": len(inside),
        "search_centers": len(points),
        "candidate_locations": len(features),
        "candidates_within_1000m": int(features.geometry.intersects(area).sum()),
    }
    return points, stats


def export_geoparquet(checkpoint, destination):
    import os
    from tempfile import NamedTemporaryFile

    import geopandas as gpd
    import pandas as pd

    checkpoint, destination = Path(checkpoint), Path(destination)
    if not checkpoint.is_file():
        raise FileNotFoundError("Сначала создайте файл прогресса сборщика.")
    with closing(sqlite3.connect(checkpoint.resolve().as_uri() + "?mode=ro", uri=True)) as db:
        rows = pd.read_sql_query(
            "SELECT * FROM organizations WHERE lon IS NOT NULL ORDER BY id", db
        )
    geometry = gpd.points_from_xy(rows.pop("lon"), rows.pop("lat"), crs="EPSG:4326")
    gdf = gpd.GeoDataFrame(rows, geometry=geometry).to_crs("EPSG:32643")
    destination.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=destination.parent, suffix=".parquet", delete=False) as temp:
        temp_path = Path(temp.name)
    try:
        gdf.to_parquet(temp_path, index=False)
        os.replace(temp_path, destination)
    finally:
        temp_path.unlink(missing_ok=True)
    return gdf

## Mock-тесты без сети

In [3]:
from datetime import datetime
from email.utils import format_datetime
import tempfile
import unittest
from unittest.mock import Mock


class CollectorTests(unittest.TestCase):
    def setUp(self):
        temp = tempfile.TemporaryDirectory()
        self.addCleanup(temp.cleanup)
        self.checkpoint = Path(temp.name) / "mock.sqlite3"
        self.points = ["76.94,43.24", "76.95,43.25"]
        self.search = Search(page_size=2, max_pages=3)
        self.clock = 1_700_000_000.0
        self.waits = []

    def sleep(self, seconds):
        self.waits.append(seconds)
        self.clock += seconds

    def run_collection(self, responses, *, limit=20, centers=2, search=None):
        get = Mock(side_effect=responses)
        self.http = get
        result = collect(
            self.points, search or self.search, self.checkpoint, key="mock-credential",
            max_requests_total=limit, center_limit=centers, get=get,
            now=lambda: self.clock, sleep=self.sleep,
        )
        self.assertTrue(all(call.args == (PLACES_URL,) for call in get.call_args_list))
        self.assertTrue(all(not call.kwargs["allow_redirects"] for call in get.call_args_list))
        return result

    @staticmethod
    def response(status=200, *, code=200, total=0, items=(), retry_after=None, error=None):
        data = {"meta": {"code": code}, "result": {"items": list(items), "total": total}}
        if error:
            data["meta"]["error"] = {"type": error}
        return Mock(status_code=status, headers={"Retry-After": retry_after},
                    **{"json.return_value": data})

    def test_retry_within_pagination_and_dedup(self):
        result = self.run_collection([
            self.response(total=3, items=[{"id": "a"}, {"id": "b"}]),
            self.response(500), self.response(code=503),
            self.response(total=3, items=[{"id": "c"}]),
            self.response(total=1, items=[{"id": "a", "point": {"lon": 76.94, "lat": 43.24}}]),
        ])
        self.assertEqual([call.kwargs["params"]["page"] for call in self.http.call_args_list],
                         [1, 2, 2, 2, 1])
        self.assertEqual(result["attempts"], 5)
        self.assertEqual(result["organizations"], 3)
        self.assertEqual(result["without_coordinates"], 2)
        self.assertEqual(result["centers"]["complete"], 2)
        self.assertIsNone(result["stop_reason"])

    def test_429_retry_after_seconds(self):
        result = self.run_collection([self.response(429, retry_after="7"), self.response()],
                                     centers=1)
        self.assertEqual(self.waits, [7])
        self.assertEqual(result["centers"]["complete"], 1)

    def test_retry_after_http_date_survives_resume(self):
        retry_at = format_datetime(
            datetime.fromtimestamp(self.clock + 90, tz=timezone.utc), usegmt=True
        )
        result = self.run_collection([self.response(429, retry_after=retry_at)])
        self.assertEqual(result["attempts"], 1)
        result = self.run_collection([])
        self.http.assert_not_called()
        self.assertEqual(result["attempts"], 1)
        self.clock += 91
        result = self.run_collection([self.response(), self.response()])
        self.assertEqual(result["centers"]["complete"], 2)

    def test_network_retries_are_bounded_and_errors_are_sanitized(self):
        result = self.run_collection([requests.exceptions.Timeout("mock-credential")] * 3)
        self.assertEqual(result["attempts"], 3)
        self.assertEqual(result["centers"]["pending"], 2)
        self.assertNotIn("mock-credential", result["stop_reason"])

    def test_permanent_errors_stop_without_retry(self):
        for response in (self.response(401), self.response(code=403), self.response(302),
                         self.response(404), self.response(code=404, error="unknown")):
            with self.subTest(status=response.status_code):
                result = self.run_collection([response])
                self.assertEqual(self.http.call_count, 1)
                self.assertEqual(result["centers"]["pending"], 2)

    def test_budget_resume_preserves_page_and_counts_retries(self):
        result = self.run_collection([
            self.response(total=3, items=[{"id": "a"}, {"id": "b"}]),
            self.response(503),
        ], limit=2)
        self.assertEqual(result["organizations"], 2)
        self.assertEqual(result["attempts"], 2)
        self.assertEqual(result["centers"]["pending"], 2)
        self.assertIn("лимит", result["stop_reason"])
        self.run_collection([], limit=2)
        self.http.assert_not_called()
        recovered = self.run_collection([
            self.response(total=3, items=[{"id": "c"}]), self.response(),
        ], limit=4)
        self.assertEqual([call.kwargs["params"]["page"] for call in self.http.call_args_list],
                         [2, 1])
        self.assertEqual(recovered["centers"]["complete"], 2)
        self.assertEqual(recovered["attempts"], 4)
        self.run_collection([], limit=4)
        self.http.assert_not_called()

    def test_empty_and_capped_centers(self):
        result = self.run_collection([
            self.response(code=404, error="itemNotFound"),
            self.response(total=100, items=[{"id": "a"}, {"id": "b"}]),
            self.response(total=100, items=[{"id": "c"}, {"id": "d"}]),
        ], search=Search(page_size=2, max_pages=2))
        self.assertEqual(result["centers"]["complete"], 1)
        self.assertEqual(result["centers"]["capped"], 1)
        self.assertEqual(result["attempts"], 3)

    def test_repeated_ids_and_early_empty_page_are_reported(self):
        result = self.run_collection([
            self.response(total=3, items=[{"id": "a"}, {"id": "b"}]),
            self.response(total=3, items=[{"id": "a"}]),
            self.response(total=10, items=[{"id": "c"}, {"id": "d"}]),
            self.response(code=404, error="itemNotFound"),
        ])
        self.assertEqual(result["centers"]["short"], 2)
        self.assertEqual(result["organizations"], 4)

    def test_bad_payloads_do_not_advance_cursor(self):
        bad_json = self.response()
        bad_json.json.side_effect = ValueError("mock-credential")
        for response in (bad_json, self.response(total=1, items=[{}])):
            result = self.run_collection([response])
            self.assertEqual(self.http.call_count, 1)
            self.assertEqual(result["centers"]["pending"], 2)
            self.assertEqual(result["organizations"], 0)
            self.assertNotIn("mock-credential", result["stop_reason"])
        self.assertEqual(self.http.call_args.kwargs["params"]["page"], 1)

    def test_changed_plan_and_concurrent_writer_are_rejected(self):
        self.run_collection([self.response()], centers=1)
        with self.assertRaisesRegex(CollectionStopped, "План поиска изменился"):
            self.run_collection([], search=Search(query="пекарня"))
        self.http.assert_not_called()
        with open_checkpoint(self.checkpoint):
            with self.assertRaisesRegex(CollectionStopped, "уже используется"):
                self.run_collection([])
            self.http.assert_not_called()


suite = unittest.defaultTestLoader.loadTestsFromTestCase(CollectorTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Проверки сборщика 2ГИС завершились ошибкой.")

test_429_retry_after_seconds (__main__.CollectorTests.test_429_retry_after_seconds) ... ok
test_bad_payloads_do_not_advance_cursor (__main__.CollectorTests.test_bad_payloads_do_not_advance_cursor) ... ok
test_budget_resume_preserves_page_and_counts_retries (__main__.CollectorTests.test_budget_resume_preserves_page_and_counts_retries) ... ok
test_changed_plan_and_concurrent_writer_are_rejected (__main__.CollectorTests.test_changed_plan_and_concurrent_writer_are_rejected) ... ok
test_empty_and_capped_centers (__main__.CollectorTests.test_empty_and_capped_centers) ... ok
test_network_retries_are_bounded_and_errors_are_sanitized (__main__.CollectorTests.test_network_retries_are_bounded_and_errors_are_sanitized) ... ok
test_permanent_errors_stop_without_retry (__main__.CollectorTests.test_permanent_errors_stop_without_retry) ... ok
test_repeated_ids_and_early_empty_page_are_reported (__main__.CollectorTests.test_repeated_ids_and_early_empty_page_are_reported) ... ok
test_retry_after_http_da

## Настройки и предварительный расчёт

Откройте ноутбук из каталога проекта или его папки `notebooks`. Для другого расположения укажите полный путь в `PROJECT_ROOT`. Проверьте число центров и остаток квоты в Менеджере платформы 2ГИС. Параметры по умолчанию позволяют запросить до пяти страниц по десять записей на центр, что соответствует пределам демо-ключа по пагинации. Для 413 центров это от 413 до 2065 попыток без повторов.

Сначала проверьте первые три центра. Укажите согласованный суммарный бюджет в `MAX_REQUESTS_TOTAL`, затем включите `RUN_LIVE`. Если пилот прошёл успешно, проверьте координаты, рейтинги и статусы центров. Чтобы продолжить сбор, увеличьте `CENTER_LIMIT` и при необходимости суммарный бюджет. Значение бюджета включает потраченные в пилоте попытки. `capped` означает, что центр достиг предела страниц, `short` — что получено меньше уникальных ID, чем заявлено в `total`, `pending` — что центр ещё не обработан до конца.


In [11]:
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
PROJECT_ROOT = PROJECT_ROOT.resolve()

SEARCH = Search()
RUN_LIVE = False
CENTER_LIMIT = 20
MAX_REQUESTS_TOTAL = 100
MIN_INTERVAL_SECONDS = 1.0
CHECKPOINT = PROJECT_ROOT / "data" / "processed" / "2gis_almaty_progress.sqlite3"
EXPORT = PROJECT_ROOT / "data" / "processed" / "2gis_almaty_coffee.parquet"

search_points = None
if (PROJECT_ROOT / "data" / "processed" / "almaty_features.parquet").is_file():
    search_points, grid_summary = prepare_almaty_grid(PROJECT_ROOT)
    print(grid_summary)
    print(f"Верхняя оценка без повторов: {len(search_points) * SEARCH.max_pages}")
else:
    print("Укажите PROJECT_ROOT, чтобы загрузить локальные данные и проверить сетку.")

{'grid_inside_almaty': 703, 'search_centers': 413, 'candidate_locations': 2822, 'candidates_within_1000m': 1950}
Верхняя оценка без повторов: 2065


## Сбор по 2ГИС

Задайте переменную окружения `DGIS_API_KEY` перед запуском Jupyter. Ключ считывается только при включённом сборе. Если 2ГИС потребовал подождать, время следующей попытки сохраняется в файле прогресса.


In [12]:
def run_live_collection(points):
    import os

    if points is None or len(points) != 413:
        raise ValueError("Проверьте сетку. Для этого запуска ожидается 413 центров.")
    if type(MAX_REQUESTS_TOTAL) is not int or MAX_REQUESTS_TOTAL < 1:
        raise ValueError("Укажите положительный MAX_REQUESTS_TOTAL после проверки квоты.")
    key = os.environ.get("DGIS_API_KEY", "").strip()
    if not key:
        raise ValueError("Задайте переменную окружения DGIS_API_KEY для ядра Jupyter.")
    with requests.Session() as session:
        return collect(
            points, SEARCH, CHECKPOINT, key=key, get=session.get,
            max_requests_total=MAX_REQUESTS_TOTAL, center_limit=CENTER_LIMIT,
            interval=MIN_INTERVAL_SECONDS,
        )


if RUN_LIVE:
    print(run_live_collection(search_points))
else:
    print("Сбор по API выключен.")

Сбор по API выключен.


## Экспорт для геоанализа

Включите `EXPORT_RESULTS`, когда нужно обновить GeoParquet. Геометрия экспортируется в `EPSG:32643`. Организации без валидных координат остаются в SQLite. Экспорт включает все сохранённые страницы, в том числе от ещё не завершённых центров. Перед расчётом признаков проверьте сводку сбора.


In [6]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    unique_coffee_gdf = export_geoparquet(CHECKPOINT, EXPORT)
    print(f"Экспортировано организаций: {len(unique_coffee_gdf)}")
    print(f"CRS: {unique_coffee_gdf.crs}")